# E-commerce Sales - Data Cleaning

**Objective:** Clean and prepare the raw data for analysis.

**Input:** Raw transaction data  
**Output:** Cleaned dataset ready for analysis

**Author:** Hamdi Ahmed 
**Date:** 5/07/2026

# 1. Setup

In [28]:
import duckdb
import pandas as pd

df1 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
raw = pd.concat([df1, df2], ignore_index=True)


print(f"Loaded {len(raw):,} rows")

Loaded 1,067,371 rows


# 2. Standardise Column Names

In [29]:
raw = raw.rename(columns={'Customer ID': 'CustomerID'})

con = duckdb.connect()
con.register('raw_retail', raw)

print("Columns:", list(raw.columns))
print(f"Rows: {len(raw):,}")

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country']
Rows: 1,067,371


## • Building CTE chain to clean raw data in stages, with each stage fixing one issue found in audit. 
## • Some observations will have more than 1 of the data quality issues identified. 
## • Before each stage, the number of data quality issues remaining will be counted to check if they have been removed in the previous stage

# 3. Remove Exact Duplicates

In [30]:
# Audit found 34,335 exact duplicate rows where all columns including the InvoiceDate timestamp are identical
# Likely data extraction issue as same transaction can't have happened twice.

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    )
    SELECT COUNT(*) AS no_duplication_total FROM deduped;
""").fetchdf()


,no_duplication_total
0,1033036


# 4. Flag Cancellations

#### Audit found 19,494 cancellations with C-prefix in invoice and 22,950 with negative quantities.

In [32]:
# Remaining data quality issues
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * FROM raw_retail
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' OR Quantity < 0 THEN 1 ELSE 0 END) AS cancellations,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM deduped;
""").fetchdf()

,total_rows,cancellations,invalid_price,null_customer_id,null_description
0,1033036,22497.0,6019.0,235151.0,4275.0


In [33]:
# Flag Cancellations
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    )
    SELECT COUNT(*) AS rows_after_tag FROM cancellation_tag;
""").fetchdf()

,rows_after_tag
0,1033036


In [36]:
# Remaining data quality issues
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' OR Quantity < 0 THEN 1 ELSE 0 END) AS flagged_cancellations,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM cancellation_tag;
""").fetchdf()

,total_rows,flagged_cancellations,invalid_price,null_customer_id,null_description
0,1033036,22497.0,6019.0,235151.0,4275.0


# 5. Remove zero/negative prices

#### Audit found 6,207 with zero/negative prices

In [38]:
# remove zero/negativ prices

con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    )
    SELECT COUNT(*) AS rows_after_filter FROM base_transactions;
""").fetchdf()

,rows_after_filter
0,1027017


In [40]:
# Remaining data quality issues
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' OR Quantity < 0 THEN 1 ELSE 0 END) AS flagged_cancellations,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM base_transactions;
""").fetchdf()

,total_rows,flagged_cancellations,invalid_price,null_customer_id,null_description
0,1027017,19104.0,0.0,229202.0,0.0


### Removing negative/zero prices also removed rows with null descriptions

# 6. Stock Code Cleaning

#### • Stock Code usually 5 or 6 characters long, composed of numbers and letters
#### • Will remove stock codes with less than 5 characters and also look closer at stock codes with no numbers

In [41]:
# Stock Code Distribution
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    )
    SELECT
        LENGTH(CAST(StockCode AS VARCHAR)) AS code_length,
        COUNT(*) AS rows,
        COUNT(DISTINCT StockCode) AS distinct_codes,
        ANY_VALUE(StockCode) AS example
    FROM 
        base_transactions
    GROUP BY 
        code_length
    ORDER BY 
        rows DESC
""").fetchdf()

,code_length,rows,distinct_codes,example
0,5,897765,3401,47580
1,6,122113,1488,85039A
2,4,2113,3,POST
3,1,1660,5,M
4,3,1418,1,DOT
5,7,1336,8,15056BL
6,2,274,1,C2
7,12,178,8,gift_0001_50
8,8,99,14,DCGS0058
9,9,61,3,AMAZONFEE


In [42]:
# Stock Codes without numbers
con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    )
    SELECT
        CAST(StockCode AS VARCHAR) AS code,
        ANY_VALUE(Description) AS description,
        COUNT(*) AS rows,
        ROUND(MIN(Price), 2) AS min_price,
        ROUND(MAX(Price), 2) AS max_price
    FROM base_transactions
    WHERE NOT REGEXP_MATCHES(CAST(StockCode AS VARCHAR), '[0-9]')
    GROUP BY code
    """).fetchdf()

,code,description,rows,min_price,max_price
0,S,SAMPLES,101,2.80,605.18
1,B,Adjust bad debt,1,11062.06,11062.06
2,PADS,PADS TO MATCH ALL CUSHIONS,18,0.00,36.60
3,CRUK,CRUK Commission,16,1.60,1100.44
4,DOT,DOTCOM POSTAGE,1418,0.35,4505.17
5,ADJUST,Adjustment by john on 26/01/2010 17,67,4.57,5117.03
6,POST,POSTAGE,2079,0.50,8142.75
7,BANK CHARGES,Bank Charges,100,0.00,18910.69
8,AMAZONFEE,AMAZON FEE,36,1.00,17836.46
9,M,Manual,1380,0.03,38970.00


In [43]:
stock_code_cleaned = con.execute("""
    WITH deduped AS (
        SELECT DISTINCT * 
        FROM raw_retail
    ),
    cancellation_tag AS (
        SELECT *,
            CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                 OR Quantity < 0 
                 THEN 1 ELSE 0 END AS is_cancellation
        FROM deduped
    ),
    base_transactions AS (
        SELECT * 
        FROM cancellation_tag
        WHERE Price > 0
    )
    SELECT *
       FROM 
           (SELECT *
               FROM base_transactions
               WHERE
                   LENGTH(StockCode) >=5
            )
    WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
    """).fetchdf()

In [44]:
con.execute("""
   SELECT
        LENGTH(CAST(StockCode AS VARCHAR))  AS code_length,
        COUNT(*)                            AS rows,
        COUNT(DISTINCT StockCode)           AS distinct_codes,
        ANY_VALUE(StockCode)                AS example
    FROM stock_code_cleaned
    GROUP BY code_length
    ORDER BY rows DESC
""").fetchdf()

,code_length,rows,distinct_codes,example
0,5,897765,3401,21430
1,6,122046,1487,85123A
2,7,1336,8,15056BL
3,8,99,14,DCGS0003
4,12,78,7,gift_0001_30
5,9,25,2,DCGSSGIRL


# 7. Tag Guests

#### • 228,488 observations (22.7% of total) remaining have no CustomerID, their revenue is valid but will be filtered out for the customer analysis
#### • Null values in Description were found in audit but have been removed in stage 5

In [45]:
# Tag rows with no CustomerID as guest transactions
con.execute("""
    WITH guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM stock_code_cleaned
    )
    SELECT
        SUM(is_guest) AS guest_rows,
        SUM(1 - is_guest) AS registered_rows,
        COUNT(*) AS total_rows
    FROM guest_tag;
    """).fetchdf()

,guest_rows,registered_rows,total_rows
0,227167.0,794182.0,1021349


In [46]:
# Remaining data quality issues

con.execute("""
    WITH guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM stock_code_cleaned
    )
    SELECT
        COUNT(*) total_rows,
        SUM(CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' OR Quantity < 0 THEN 1 ELSE 0 END) AS flagged_cancellations,
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS invalid_price,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
        SUM(is_guest) AS guest_total,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS null_description
    FROM guest_tag;
""").fetchdf()

,total_rows,flagged_cancellations,invalid_price,null_customer_id,guest_total,null_description
0,1021349,17919.0,0.0,227167.0,227167.0,0.0


# 8. Derive Revenue

In [47]:
con.execute("""
    WITH guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM stock_code_cleaned
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    )
    SELECT
        MIN(Revenue) AS min_revenue,
        MAX(Revenue) AS max_revenue,
        ROUND(AVG(Revenue), 2) AS avg_revenue,
        ROUND(SUM(Revenue), 2) AS total_revenue,
        MIN(Quantity) AS min_quantity,
        MAX(Quantity) AS max_quantity,
        ROUND(AVG(Quantity), 2) AS avg_quantity,
        ROUND(SUM(Quantity), 2) AS total_quantity,
        MIN(Price) AS min_price,
        MAX(Price) AS max_price,
        ROUND(AVG(Price), 2) AS avg_price,
        ROUND(SUM(Price), 2) AS total_price,
        COUNT(*) total_rows
    FROM revenue_derive;
    """).fetchdf()

,min_revenue,max_revenue,avg_revenue,total_revenue,min_quantity,max_quantity,avg_quantity,total_quantity,min_price,max_price,avg_price,total_price,total_rows
0,-168469.6,168469.6,18.53,18930056.81,-80995,80995,10.5,10720433.0,0.03,1157.15,3.36,3436057.14,1021349


# 9. Date Features

In [48]:
con.execute("""
    WITH guest_tag AS (
        SELECT *,
        CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
        FROM stock_code_cleaned
    ),
    revenue_derive AS (
        SELECT 
            *,
            Quantity * Price AS Revenue
        FROM guest_tag
    ),
    cleaned AS (
        SELECT *,
            YEAR(InvoiceDate) AS Year,
            MONTH(InvoiceDate) AS Month,
            DAYOFWEEK(InvoiceDate) AS DayOfWeek,
            HOUR(InvoiceDate) AS Hour,
            CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
            DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
            DATE_TRUNC('month', InvoiceDate) AS MonthStart
        FROM revenue_derive
    )
    SELECT * FROM cleaned LIMIT 5;
    """).fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,is_cancellation,is_guest,Revenue,Year,Month,DayOfWeek,Hour,IsWeekend,WeekStart,MonthStart
0,528612,47580,TEA TIME DES TEA COSY,6,2010-10-22 15:57:00,2.55,14728.0,United Kingdom,0,0,15.30,2010,10,5,15,0,2010-10-18,2010-10-01
1,528612,21429,RED GINGHAM ROSE JEWELLERY BOX,6,2010-10-22 15:57:00,1.65,14728.0,United Kingdom,0,0,9.90,2010,10,5,15,0,2010-10-18,2010-10-01
2,528612,22111,SCOTTIE DOG HOT WATER BOTTLE,3,2010-10-22 15:57:00,4.95,14728.0,United Kingdom,0,0,14.85,2010,10,5,15,0,2010-10-18,2010-10-01
3,528613,22867,HAND WARMER BIRD DESIGN,1,2010-10-22 15:57:00,4.21,NaN,United Kingdom,0,1,4.21,2010,10,5,15,0,2010-10-18,2010-10-01
4,528613,22903,CALENDAR FAMILY FAVOURITES,1,2010-10-22 15:57:00,5.91,NaN,United Kingdom,0,1,5.91,2010,10,5,15,0,2010-10-18,2010-10-01


# 8. Validate Final Dataset

In [51]:
# Counting number of observations per data cleaning step

con.execute("""
    WITH deduped AS (
            SELECT DISTINCT * 
            FROM raw_retail
        ),
        cancellation_tag AS (
            SELECT *,
                CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                     OR Quantity < 0 
                     THEN 1 ELSE 0 END AS is_cancellation
            FROM deduped
        ),
        base_transactions AS (
            SELECT * 
            FROM cancellation_tag
            WHERE Price > 0
        ),
        stock_code_cleaned AS (
            SELECT *
               FROM 
                   (SELECT *
                       FROM base_transactions
                       WHERE
                           LENGTH(StockCode) >=5
                    )
            WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
        ),
        guest_tag AS (
            SELECT *,
            CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
            FROM stock_code_cleaned
        ),
        revenue_derive AS (
            SELECT 
                *,
                Quantity * Price AS Revenue
            FROM guest_tag
        ),
        cleaned AS (
            SELECT *,
                YEAR(InvoiceDate) AS Year,
                MONTH(InvoiceDate) AS Month,
                DAYOFWEEK(InvoiceDate) AS DayOfWeek,
                HOUR(InvoiceDate) AS Hour,
                CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
                DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
                DATE_TRUNC('month', InvoiceDate) AS MonthStart
            FROM revenue_derive
        )
    SELECT 'raw' AS stage, COUNT(*) AS num_of_rows FROM raw_retail
    UNION ALL SELECT 'deduped', COUNT(*) FROM deduped
    UNION ALL SELECT 'flagged_cancellations', COUNT(*) FROM cancellation_tag
    UNION ALL SELECT 'base_transactions', COUNT(*) FROM base_transactions
    UNION ALL SELECT 'stock_code_cleaned', COUNT(*) FROM stock_code_cleaned
    UNION ALL SELECT 'with_guest_tag', COUNT(*) FROM guest_tag
    UNION ALL SELECT 'with_revenue', COUNT(*) FROM revenue_derive
    UNION ALL SELECT 'cleaned', COUNT(*) FROM cleaned;
    """).fetchdf()

,stage,num_of_rows
0,raw,1067371
1,deduped,1033036
2,flagged_cancellations,1033036
3,base_transactions,1027017
4,stock_code_cleaned,1021349
5,with_guest_tag,1021349
6,with_revenue,1021349
7,cleaned,1021349


In [54]:
# Checking if any data quality issues still exist
con.execute("""
    WITH deduped AS (
            SELECT DISTINCT * 
            FROM raw_retail
        ),
        cancellation_tag AS (
            SELECT *,
                CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                     OR Quantity < 0 
                     THEN 1 ELSE 0 END AS is_cancellation
            FROM deduped
        ),
        base_transactions AS (
            SELECT * 
            FROM cancellation_tag
            WHERE Price > 0
        ),
        stock_code_cleaned AS (
            SELECT *
                FROM 
                    (SELECT *
                        FROM base_transactions
                        WHERE
                            LENGTH(StockCode) >= 5
                    )
            WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
        ),
        guest_tag AS (
            SELECT *,
            CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
            FROM stock_code_cleaned
        ),
        revenue_derive AS (
            SELECT 
                *,
                Quantity * Price AS Revenue
            FROM guest_tag
        ),
        cleaned AS (
            SELECT *,
                YEAR(InvoiceDate) AS Year,
                MONTH(InvoiceDate) AS Month,
                DAYOFWEEK(InvoiceDate) AS DayOfWeek,
                HOUR(InvoiceDate) AS Hour,
                CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
                DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
                DATE_TRUNC('month', InvoiceDate) AS MonthStart
            FROM revenue_derive
        )
    SELECT
        SUM(CASE WHEN Price <= 0 THEN 1 ELSE 0 END) AS bad_price,
        SUM(CASE WHEN is_cancellation  NOT IN (0, 1) THEN 1 ELSE 0 END) AS bad_cancellation_flag,
        SUM(CASE WHEN LENGTH(StockCode) < 5 
                   OR StockCode IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE') THEN 1 ELSE 0 END) AS bad_stock_code,
        SUM(CASE WHEN IsWeekend NOT IN (0, 1) THEN 1 ELSE 0 END) AS bad_weekend_flag,
        SUM(CASE WHEN is_guest  NOT IN (0, 1) THEN 1 ELSE 0 END) AS bad_guest_flag
    FROM cleaned;
    """).fetchdf()

,bad_price,bad_cancellation_flag,bad_stock_code,bad_weekend_flag,bad_guest_flag
0,0.0,0.0,0.0,0.0,0.0


# 9. Saving cleaned data

In [55]:
con.execute("""
    CREATE OR REPLACE VIEW cleaned AS
    WITH deduped AS (
            SELECT DISTINCT * 
            FROM raw_retail
        ),
        cancellation_tag AS (
            SELECT *,
                CASE WHEN CAST(Invoice AS VARCHAR) LIKE 'C%' 
                     OR Quantity < 0 
                     THEN 1 ELSE 0 END AS is_cancellation
            FROM deduped
        ),
        base_transactions AS (
            SELECT * 
            FROM cancellation_tag
            WHERE Price > 0
        ),
        stock_code_cleaned AS (
            SELECT *
                FROM 
                    (SELECT *
                        FROM base_transactions
                        WHERE
                            LENGTH(StockCode) >= 5
                    )
            WHERE StockCode NOT IN ('BANK CHARGES', 'ADJUST', 'AMAZONFEE')
        ),
        guest_tag AS (
            SELECT *,
            CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END AS is_guest
            FROM stock_code_cleaned
        ),
        revenue_derive AS (
            SELECT 
                *,
                Quantity * Price AS Revenue
            FROM guest_tag
        )
    SELECT *,
        YEAR(InvoiceDate) AS Year,
        MONTH(InvoiceDate) AS Month,
        DAYOFWEEK(InvoiceDate) AS DayOfWeek,
        HOUR(InvoiceDate) AS Hour,
        CASE WHEN DAYOFWEEK(InvoiceDate) IN (0, 6) THEN 1 ELSE 0 END AS IsWeekend,
        DATE_TRUNC('week',  InvoiceDate) AS WeekStart,
        DATE_TRUNC('month', InvoiceDate) AS MonthStart
    FROM revenue_derive;
""")

In [56]:
# All valid sales data including guests
con.execute("""
    COPY (SELECT * FROM cleaned)
    TO '../data/processed/cleaned.csv' (HEADER, DELIMITER ',')
""")

In [57]:
# Registered customers only
con.execute("""
    COPY (SELECT * FROM cleaned WHERE is_guest = 0)
    TO '../data/processed/registered_only.csv' (HEADER, DELIMITER ',');
""")

# 10. Cleaning Summary

In [59]:
stats = con.execute("""
    SELECT
        COUNT(*) AS final_rows,
        COUNT(DISTINCT CustomerID) AS unique_customers,
        COUNT(DISTINCT StockCode) AS unique_products,
        COUNT(DISTINCT Country) AS unique_countries,
        ROUND(SUM(Revenue), 2) AS total_revenue
    FROM cleaned
""").fetchdf()

raw_rows    = len(raw)
final_rows  = stats['final_rows'][0]
removed     = raw_rows - final_rows
pct_removed = removed / raw_rows * 100
customers   = stats['unique_customers'][0]
products    = stats['unique_products'][0]
countries   = stats['unique_countries'][0]
revenue     = stats['total_revenue'][0]

print('=' * 60)
print('CLEANING SUMMARY')
print('=' * 60)
print(f"""
ROWS:
  Original:  {raw_rows:,}
  Final:     {final_rows:,}
  Removed:   {removed:,} ({pct_removed:.1f}%)

WHAT WAS REMOVED AND WHY:
  • Exact row duplicated, likely a data extraction issue as same transaction can't have happened twice or more at the same time down to the second.
  • Cancellations represented with 'C' prefix or negative quantities were flagged
  • Zero price removed as these were bad debt entries and negative price removed due to no clear meaning to them

FINAL DATASET (cleaned.csv):
  - Rows: {final_rows:,}
  - Customers: {customers:,} (registered only, guests excluded from count)
  - Products: {products:,}
  - Countries: {countries:,}
  - Total Revenue: GBP {revenue:,.2f}

OUTPUT FILES:
  • cleaned.csv ---> all valid sales including guests
  • registered_only.csv ---> valid sales with CustomerID for cohort and RFM

Ready for analysis.ipynb
""")

CLEANING SUMMARY

ROWS:
  Original:  1,067,371
  Final:     1,021,349
  Removed:   46,022 (4.3%)

WHAT WAS REMOVED AND WHY:
  • Exact row duplicated, likely a data extraction issue as same transaction can't have happened twice or more at the same time down to the second.
  • Cancellations represented with 'C' prefix or negative quantities were flagged
  • Zero price removed as these were bad debt entries and negative price removed due to no clear meaning to them

FINAL DATASET (cleaned.csv):
  - Rows: 1,021,349
  - Customers: 5,875 (registered only, guests excluded from count)
  - Products: 4,919
  - Countries: 43
  - Total Revenue: GBP 18,930,056.81

OUTPUT FILES:
  • cleaned.csv ---> all valid sales including guests
  • registered_only.csv ---> valid sales with CustomerID for cohort and RFM

Ready for analysis.ipynb

